# Erstellen, testen und deployen eines Agenten mit dem Mosaic-AI-Agent-Framework


Link zum Tutorial: 
[AWS](https://docs.databricks.com/aws/en/generative-ai/guide/introduction-generative-ai-apps#what-are-gen-ai-apps) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/guide/introduction-generative-ai-apps#what-are-gen-ai-apps) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/guide/introduction-generative-ai-apps)
 
Mosaic AI Agent Framework: [AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/build-genai-apps#-mosaic-ai-agent-framework) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/build-genai-apps#-mosaic-ai-agent-framework) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/build-genai-apps#-mosaic-ai-agent-framework)

In [0]:
%run "./Helper/_config"

## Einen Agenten erstellen und testen

Dieser Abschnitt definiert und testet einen einfachen Agenten mit den folgenden Eigenschaften:

- Der Agent verwendet ein LLM, das über die Databricks Foundation Model API bereitgestellt wird
- Der Agent hat Zugriff auf ein einziges Tool: das integrierte Python-Code-Interpreter-Tool im Databricks Unity Catalog. Er kann dieses Tool nutzen, um vom LLM generierten Code auszuführen und damit Nutzerfragen zu beantworten
- Wir verwenden das databricks_openai-SDK, um das LLM-Endpoint abzufragen.

In [0]:
%pip install -U -qqqq mlflow databricks-openai databricks-agents
dbutils.library.restartPython()

In [0]:
# Der folgende Codeausschnitt versucht, aus einer Menge von Kandidaten
# die erste in deinem Databricks-Workspace verfügbare LLM-API auszuwählen.
# Man kann ihn überschreiben und vereinfachen, indem man einfach den LLM_ENDPOINT_NAME direkt angibt.

LLM_ENDPOINT_NAME = None

from databricks.sdk import WorkspaceClient
def is_endpoint_available(endpoint_name):
  try:
    client = WorkspaceClient().serving_endpoints.get_open_ai_client()
    client.chat.completions.create(model=endpoint_name, messages=[{"role": "user", "content": "What is AI?"}])
    return True
  except Exception:
    return False
  
client = WorkspaceClient()
for candidate_endpoint_name in ["databricks-claude-3-7-sonnet", "databricks-meta-llama-3-3-70b-instruct"]:
    if is_endpoint_available(candidate_endpoint_name):
      LLM_ENDPOINT_NAME = candidate_endpoint_name
      print(LLM_ENDPOINT_NAME)
assert LLM_ENDPOINT_NAME is not None, "Please specify LLM_ENDPOINT_NAME"

In [0]:
import json
import mlflow
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient

# Logge automatisch Traces aus LLM-Aufrufen, um das Debuggen zu erleichtern
mlflow.openai.autolog()

# OpenAI-Client abrufen, der für die Kommunikation mit Databricks-Model-Serving-Endpunkten konfiguriert ist
# Diesen verwenden wir, um in unserem Agenten ein LLM abzufragen

openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Databricks built-in tools laden (ein zustandsloses Python-Code-Interpreter-Tool)

client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(
    function_names=["system.ai.python_exec"], client=client
).tools
for tool in builtin_tools:
    del tool["function"]["strict"]


def call_tool(tool_name, parameters):
    if tool_name == "system__ai__python_exec":
        return DatabricksFunctionClient().execute_function(
            "system.ai.python_exec", parameters=parameters
        )
    raise ValueError(f"Unknown tool: {tool_name}")


def run_agent(prompt):
    """
    Send a user prompt to the LLM, and return a list of LLM response messages
    The LLM is allowed to call the code interpreter tool if needed, to respond to the user
    """
    result_msgs = []
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
        tools=builtin_tools,
    )
    msg = response.choices[0].message
    result_msgs.append(msg.to_dict())

    # Falls das Modell ein Tool ausgeführt hat, rufe es auf
    if msg.tool_calls:
        call = msg.tool_calls[0]
        tool_result = call_tool(call.function.name, json.loads(call.function.arguments))
        result_msgs.append(
            {
                "role": "tool",
                "content": tool_result.value,
                "name": call.function.name,
                "tool_call_id": call.id,
            }
        )
    return result_msgs

In [0]:
answer = run_agent("What is the square root of 429?")
for message in answer:
    print(f'{message["role"]}: {message["content"]}')

In [0]:

from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

CATALOG = f"{CATALOG}"
SCHEMA = f"{SCHEMA}"

def your_function():
  ### YOUR CODE HERE ###
  
  return

function_info = client.create_python_function(
  func=your_function,
  catalog=CATALOG,
  schema=SCHEMA,
  replace=True
)

## Agent-Code für das Logging vorbereiten

Wrap deine Agent Definition in die ChatAgent-Schnittstelle von MLflow [ChatAgent interface](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ChatAgent) ein, um deinen Code für das Logging vorzubereiten.

Durch die Nutzung der standardisierten Agent-Authoring-Schnittstelle von MLflow erhältst du integrierte Benutzeroberflächen, um mit deinem Agenten zu chatten und ihn nach der Bereitstellung mit anderen zu teilen ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent#-use-chatagent-to-author-agents) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/author-agent) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/author-agent))

In [0]:
import uuid
import mlflow
from typing import Any, Optional

from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse, ChatContext

mlflow.openai.autolog()

class QuickstartAgent(ChatAgent):
    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        # 1. Den letzten Benutzerprompt aus den Eingaben extrahieren
        prompt = messages[-1].content

        # 2. Rufe run_agent auf, um eine Liste von Antwortnachrichten zurückzubekommen.
        raw_msgs = run_agent(prompt)

        # 3. Jede Antwortnachricht in ein ChatAgentMessage umwandeln und zurückgeben.
        out = []
        for m in raw_msgs:
            out.append(ChatAgentMessage(id=uuid.uuid4().hex, **m))

        return ChatAgentResponse(messages=out)

In [0]:
AGENT = QuickstartAgent()
for response_message in AGENT.predict(
    {"messages": [{"role": "user", "content": "What's the square root of 429?"}]}
).messages:
    print(f"role: {response_message.role}, content: {response_message.content}")

## Agent Logging

Logge den Agenten und registriere ihn als Modell im Unity Catalog ([AWS](https://docs.databricks.com/aws/en/machine-learning/manage-model-lifecycle/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/manage-model-lifecycle/) | [GCP](https://docs.databricks.com/gcp/en/machine-learning/manage-model-lifecycle/)). Dieser Schritt verpackt den Agenten-Code und dessen Abhängigkeiten in ein einziges Artefakt, um ihn an einem Serving-Endpunkt bereitzustellen.

Die folgenden Codezellen machen Folgendes:

1. Kopieren den oben definierten Agenten-Code und fassen ihn in einer einzelnen Zelle zusammen.
2. Fügen am Anfang der Zelle den Cell-Magic-Befehl `%%writefile` hinzu, um den Agenten-Code in einer Datei namens `quickstart_agent.py` zu speichern.
3. Fügen am Ende der Zelle einen Aufruf von [mlflow.models.set_model()](https://mlflow.org/docs/latest/model#models-from-code) hinzu. Dieser gibt MLflow an, welches Python-Agentenobjekt für Vorhersagen verwendet werden soll, wenn dein Agent bereitgestellt wird.
4. Loggen den Agenten-Code in der Datei quickstart_agent.py mithilfe der MLflow-APIs ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/log-agent) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/log-agent) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/log-agent))).

In [0]:
%%writefile quickstart_agent.py

import json
import uuid
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient
from typing import Any, Optional

import mlflow
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse, ChatContext

# OpenAI-Client abrufen, der für die Kommunikation mit Databricks-Model-Serving-Endpunkten konfiguriert ist
# Diesen verwenden wir, um in unserem Agenten ein LLM abzufragen
openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Der folgende Codeausschnitt versucht, aus einer Menge von Kandidaten
# die erste in deinem Databricks-Workspace verfügbare LLM-API auszuwählen.
# Man kann ihn überschreiben und vereinfachen, indem man einfach den LLM_ENDPOINT_NAME direkt angibt.
LLM_ENDPOINT_NAME = None

def is_endpoint_available(endpoint_name):
  try:
    client = WorkspaceClient().serving_endpoints.get_open_ai_client()
    client.chat.completions.create(model=endpoint_name, messages=[{"role": "user", "content": "What is AI?"}])
    return True
  except Exception:
    return False
  
for candidate_endpoint_name in ["databricks-claude-3-7-sonnet", "databricks-meta-llama-3-3-70b-instruct"]:
    if is_endpoint_available(candidate_endpoint_name):
      LLM_ENDPOINT_NAME = candidate_endpoint_name
assert LLM_ENDPOINT_NAME is not None, "Please specify LLM_ENDPOINT_NAME"

# Logge automatisch Traces aus LLM-Aufrufen, um das Debuggen zu erleichtern
mlflow.openai.autolog()

# Databricks built-in tools laden (ein zustandsloses Python-Code-Interpreter-Tool)
client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(function_names=["system.ai.python_exec"], client=client).tools
for tool in builtin_tools:
    del tool["function"]["strict"]

def call_tool(tool_name, parameters):
    if tool_name == "system__ai__python_exec":
        return DatabricksFunctionClient().execute_function("system.ai.python_exec", parameters=parameters)
    raise ValueError(f"Unknown tool: {tool_name}")

def run_agent(prompt):
    """
    Send a user prompt to the LLM, and return a list of LLM response messages
    The LLM is allowed to call the code interpreter tool if needed, to respond to the user
    """
    result_msgs = []
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
        tools=builtin_tools,
    )
    msg = response.choices[0].message
    result_msgs.append(msg.to_dict())

    # Falls das Modell ein Tool ausgeführt hat, rufe es auf
    if msg.tool_calls:
        call = msg.tool_calls[0]
        tool_result = call_tool(call.function.name, json.loads(call.function.arguments))
        result_msgs.append({"role": "tool", "content": tool_result.value, "name": call.function.name, "tool_call_id": call.id})
    return result_msgs

class QuickstartAgent(ChatAgent):
    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        prompt = messages[-1].content
        raw_msgs = run_agent(prompt)
        out = []
        for m in raw_msgs:
            out.append(ChatAgentMessage(
                id=uuid.uuid4().hex,
                **m
            ))

        return ChatAgentResponse(messages=out)

AGENT = QuickstartAgent()
mlflow.models.set_model(AGENT)

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution
from quickstart_agent import LLM_ENDPOINT_NAME

# Registriere das Modell im Standard-Katalog des Workspaces.

print(CATALOG)
catalog_name = CATALOG #spark.sql("SELECT current_catalog()").collect()[0][0]

print(SCHEMA)
schema_name = SCHEMA

registered_model_name = f"{catalog_name}.{schema_name}.quickstart_agent"

# Gib die Databricks-Produktressourcen an, auf die der Agent zugreifen muss
# (unser integriertes Python-Code-Interpreter-Tool und LLM-Serving-Endpunkt),
# damit Databricks beim Deployment automatisch die Authentifizierung für den Agenten konfigurieren kann.
resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
    DatabricksFunction(function_name="system.ai.python_exec"),
]

mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model="quickstart_agent.py",
        extra_pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}"
        ],
        resources=resources,
        registered_model_name=registered_model_name,
    )


## Agenten deployen

Run the cell below to deploy the agent ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/deploy-agent) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/deploy-agent)). Once the agent endpoint starts, you can chat with it via AI Playground ([AWS](https://docs.databricks.com/aws/en/large-language-models/ai-playground) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-playground) | [GCP](https://docs.databricks.com/gcp/en/large-language-models/ai-playground)), or share it with stakeholders ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-evaluation/review-app) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/review-app) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-evaluation/review-app)) for initial feedback, before sharing it more broadly.

Führe die folgende Zelle aus, um den Agenten bereitzustellen ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/deploy-agent) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/deploy-agent)).

Sobald der Agent-Endpunkt aktiv ist, kannst du über den AI Playground ([AWS](https://docs.databricks.com/aws/en/large-language-models/ai-playground) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-playground) | [GCP](https://docs.databricks.com/gcp/en/large-language-models/ai-playground)) mit ihm interagieren
oder ihn zunächst Stakeholdern zur Rückmeldung freigeben
(AWS
 | Azure
 | GCP
), bevor du ihn breiter teilst.

In [0]:
from databricks import agents

deployment_info = agents.deploy(
    model_name=registered_model_name,
    model_version=logged_agent_info.registered_model_version,
    deploy_feedback_model=False,
)